# Reinforcement Learning: A First Look
### A companion to *Supply Chain AI in Python* · scmaiteaching.org

*A companion extra, written for the site. It is not part of the book.*

**License:** the code in this notebook is under the MIT License (full text in the last cell). It covers the code only, not the explanatory text. © 2026 Charles C. Luo.

Reinforcement learning (RL) is learning to make a **sequence of decisions** by trial and error, guided only by a **score** (a reward), not by examples of the right answer.

This notebook has two small experiments. Each runs in a few seconds, needs nothing installed (Colab already has numpy and matplotlib), and checks its own results.

1. **Three slot machines** — the trade-off between trying something new and sticking with what looks best.
2. **A robot in a grid** — how a reward that comes at the end teaches the steps that lead to it.

**First time in Colab?**

1. **Sign in.** If you aren't signed in to Google, choose *Sign in* (top right). If your school account can't open Colab, use a personal one.
2. **Run everything.** Choose *Runtime → Run all*. Colab warns that the notebook was not authored by Google; this one installs nothing and asks for no access to your files, so choose *Run anyway*.
3. **Wait for each cell to finish.** A green check mark appears beside a cell once it has run. The whole notebook takes under a minute.
4. **Run one cell.** Click inside it and press *Shift+Enter*. Go from top to bottom: a later cell uses what an earlier one made.
5. **If a cell fails**, for example with a `NameError` after a break (Colab clears the session when it disconnects) or after running cells out of order, choose *Runtime → Restart session and run all*.
6. **Keep your changes.** Edits to the shared notebook are not saved for you. Choose *File → Save a copy in Drive*. The copy opens in a new tab; it is yours, so keep working there, starting with *Runtime → Run all*.

Then change the settings marked `# try changing this` and run again. The checks run only on the settings as shipped; once you change one, they are skipped, because the shipped conclusions may no longer hold — that is the point of experimenting.

---
## The words you need

| Word | Meaning | With the slot machines | In the grid |
|---|---|---|---|
| **Agent** | The learner that decides | You, choosing a machine | The robot |
| **Environment** | Everything the agent acts on | The three machines | The grid |
| **State** | The situation the agent observes when it decides | Always the same: the machines never change | The robot's square |
| **Action** | What the agent decides | Which machine to play | Up, down, left or right |
| **Reward** | The score after each action | 1 if the machine pays, else 0 | −1 per step, +10 at the goal, −10 in a hole |
| **Policy** | The agent's rule for choosing | "Play the machine with the best record so far" | An arrow on every square |
| **Episode** | One complete try, start to finish | 1,000 plays | One trip |

A **run** is one complete, independent repeat of an experiment with fresh luck. Results vary from run to run, so the notebook repeats each experiment many times.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print('numpy', np.__version__, '- ready')

---
## 1. Three slot machines: explore or exploit?

Three machines pay out with different chances. **You do not know the chances**; you only see whether each play pays. Every play on a worse machine gives up some expected payout, but if you never try the others, you may never find the best one. This is the **exploration–exploitation trade-off**, and the slot-machine version is called the *multi-armed bandit*.

Four ways to play 1,000 times:

- **Random** — a baseline: pick a machine at random every time.
- **Try 20 each, then best so far** — play each machine 20 times, then always play the one with the best record so far.
- **Greedy** — play each machine once, then always play the one with the best record so far.
- **ε-greedy** — like greedy, but a share ε (epsilon, 10% here) of the time pick a machine at random, to keep checking the others.

When two machines have the same record, the tie is broken at random.

In [ ]:
TRUE_CHANCE = np.array([0.30, 0.50, 0.60])   # hidden from the player   # try changing this
PLAYS, RUNS = 1000, 500                          # plays per run; independent runs
EPSILON, TRIAL = 0.1, 20                         # try changing this

def best_so_far(wins, pulls, rng):
    rate = wins / np.maximum(pulls, 1)
    return int(rng.choice(np.flatnonzero(rate == rate.max())))   # ties broken at random

def play(strategy, rng):
    pulls = np.zeros(3); wins = np.zeros(3); got = np.zeros(PLAYS)
    for t in range(PLAYS):
        if strategy == 'random':
            arm = int(rng.integers(3))
        elif strategy == 'try 20 each, then best so far' and t < 3 * TRIAL:
            arm = t % 3
        elif strategy in ('greedy', 'epsilon-greedy') and t < 3:
            arm = t                                                   # play each machine once
        elif strategy == 'epsilon-greedy' and rng.random() < EPSILON:
            arm = int(rng.integers(3))
        else:
            arm = best_so_far(wins, pulls, rng)
        got[t] = rng.random() < TRUE_CHANCE[arm]
        pulls[arm] += 1; wins[arm] += got[t]
    return got, int(np.argmax(pulls))                 # every play's payout; the machine played most

STRATEGIES = ('random', 'try 20 each, then best so far', 'greedy', 'epsilon-greedy')
rng = np.random.default_rng(2026)
results = {s: [play(s, rng) for _ in range(RUNS)] for s in STRATEGIES}

best = TRUE_CHANCE.max()
print(f'Best possible: {best:.3f} per play on average (a lucky run can do a little better)\n')
print(f"{'Strategy':<31}{'Average':>8}{'Middle 90% of runs':>21}{'Played a worse machine most':>30}")
summary = {}
for s, out in results.items():
    avg = np.array([got.mean() for got, _ in out])
    worse = '-' if s == 'random' else f"{100 * np.mean([m != np.argmax(TRUE_CHANCE) for _, m in out]):.0f}% of runs"
    lo, hi = np.percentile(avg, [5, 95])
    summary[s] = (avg.mean(), lo, hi)
    print(f'{s:<31}{avg.mean():>8.3f}{lo:>12.3f} to {hi:.3f}{worse:>30}')
print('\nMiddle 90%: the range of the run averages once the lowest 5% and highest 5% of runs are set aside.')

SHIPPED = TRUE_CHANCE.tolist() == [0.30, 0.50, 0.60] and (PLAYS, RUNS, EPSILON, TRIAL) == (1000, 500, 0.1, 20)
if SHIPPED:
    assert summary['epsilon-greedy'][0] > summary['random'][0] + 0.05, 'epsilon-greedy should clearly beat random play'
    assert summary['greedy'][1] < summary['epsilon-greedy'][1], 'greedy should have the worse bad runs'
    print('Checks passed (settings as shipped).')
else:
    print('Checks skipped: you changed the settings.')

**What to notice** (with the settings as shipped)

- **Compare with a baseline.** A strategy's average means little alone. Next to *random* play and the best possible (0.600), it means something.
- **A simple method is a serious baseline.** *Try 20 each, then best so far* is about as simple as a rule can be, and it is competitive with ε-greedy. A cleverer method has to beat the simple one, not just the random one.
- **Greedy can get stuck.** If the best machine happens to lose its first play, greedy may never go back to it. Its bad runs are much worse than ε-greedy's: look at the left end of its middle-90% range. A little exploration costs a little payout and cuts the share of runs that mostly played a worse machine.
- **Report the spread, not one run.** The middle-90% column matters as much as the average: a method that wins on average but sometimes does badly may not be the one you want.

*Try:* set `TRUE_CHANCE` to `[0.50, 0.55, 0.60]` (machines that are hard to tell apart), or `EPSILON` to `0.3`, and run again.

In [ ]:
# The running average payout, averaged over all runs of each strategy
for s in STRATEGIES:
    curve = np.mean([np.cumsum(got) / np.arange(1, PLAYS + 1) for got, _ in results[s]], axis=0)
    plt.plot(curve, label=s)
plt.axhline(TRUE_CHANCE.max(), ls='--', c='grey', label='best possible on average')
plt.xlabel('play'); plt.ylabel('average payout so far'); plt.ylim(0.2, 0.7); plt.legend(); plt.show()

*Reading the chart:* the "try 20 each" strategy pays for its exploration up front — for its first 60 plays it earns no more than random play — and then catches up. ε-greedy keeps paying a little for exploration all the way through.

---
## 2. A robot in a grid: learning from a reward that comes at the end

A robot starts in the top-left corner of a 5×5 grid and must reach the goal in the bottom-right corner. Each step costs 1 point; reaching the goal earns 10; falling into one of the two holes costs 10 and ends the trip. Bumping into the edge leaves the robot where it was. Every move has one certain result.

```
S . . . .
. . H . .
. . . . .
. H . . .
. . . . G
```

The big reward (or penalty) comes only at the end, so the robot has to work out which early steps led to it. RL handles this with **value**: the total reward the robot can still expect from a square, if it acts as well as possible from there on. A point one step later counts a little less — 0.95 of a point here (the *discount*, `GAMMA`) — which also makes shorter routes worth more.

**The Bellman idea, in words:** the value of a square is what its best move is worth: that move's reward plus the (discounted) value of the square it leads to. At the goal or a hole the trip ends, so there is nothing after it. Value spreads backward from the goal, one square at a time.

In [ ]:
N, START, GOAL, HOLES = 5, (0, 0), (4, 4), {(1, 2), (3, 1)}
MOVES = {'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1)}
ARROW = {'up': '^', 'down': 'v', 'left': '<', 'right': '>'}
GAMMA = 0.95          # how much a point one step later counts      # try changing this
STEP_COST = -1.0      # reward for an ordinary step                 # try changing this

def step(cell, move):
    r, c = cell[0] + MOVES[move][0], cell[1] + MOVES[move][1]
    nxt = (min(max(r, 0), N - 1), min(max(c, 0), N - 1))     # the edge keeps you in place
    if nxt == GOAL:  return nxt, 10.0, True
    if nxt in HOLES: return nxt, -10.0, True
    return nxt, STEP_COST, False

cells = [(r, c) for r in range(N) for c in range(N) if (r, c) != GOAL and (r, c) not in HOLES]

# (a) We know the rules, so we can compute the values: repeat the Bellman step until nothing changes
V = {cell: 0.0 for cell in cells}
for sweep in range(1000):
    change = 0.0
    for cell in cells:
        best = max(rew + (0 if done else GAMMA * V[nxt]) for nxt, rew, done in (step(cell, m) for m in MOVES))
        change = max(change, abs(best - V[cell])); V[cell] = best
    if change < 1e-9: break

def greedy_move(cell, value_of):
    return max(MOVES, key=lambda m: value_of(cell, m))

def trip(choose, limit=50):
    # Follow a policy from S; return (steps, where it ended)
    cell, steps = START, 0
    while steps < limit:
        cell, _, done = step(cell, choose(cell)); steps += 1
        if done: return steps, ('goal' if cell == GOAL else 'hole')
    return None, 'never ended'

exact = lambda cell, m: (lambda nxt, rew, done: rew + (0 if done else GAMMA * V[nxt]))(*step(cell, m))
def show(choose):
    for r in range(N):
        print(' '.join('G' if (r, c) == GOAL else 'H' if (r, c) in HOLES else ARROW[choose((r, c))] for c in range(N)))

print(f'Value iteration settled after {sweep + 1} sweeps. Values (rounded):')
for r in range(N):
    print(' '.join(' G  ' if (r, c) == GOAL else ' H  ' if (r, c) in HOLES else f'{V[(r, c)]:5.1f}' for c in range(N)))
print('\nBest move on every square:')
show(lambda cell: greedy_move(cell, exact))
shortest, ended = trip(lambda cell: greedy_move(cell, exact))
print(f'\nFrom S the best policy ends at the {ended} after {shortest} steps.')
SHIPPED_GRID = (GAMMA, STEP_COST) == (0.95, -1.0)
if SHIPPED_GRID:
    assert (shortest, ended) == (8, 'goal'), 'with the shipped settings the best route is 8 steps to the goal'
    print('Check passed: 8 steps, the fewest possible. Several different 8-step routes tie.')
else:
    print('Check skipped: you changed the settings.')

**(b) Now pretend we do not know the rules.** The robot only gets to *try* moves and see what happens. **Q-learning** keeps a score $Q$ for every square and move — "how good is this move from here?" — and after each step nudges it toward what that step suggests:

$$Q(\text{square}, \text{move}) \leftarrow Q + \alpha\,\big[\,\text{reward} + \gamma \max_{\text{next move}} Q(\text{next square}, \cdot) - Q\,\big]$$

In words: the bracket is what this step suggests the score should be — the reward, plus the discounted best score of the next square — minus the current score; $\alpha$ is how far to move toward it, and $\gamma$ is the same discount (`GAMMA`). At the goal or a hole there is no next square, so that term is 0.

The robot explores in two ways. It makes a random move 10% of the time (ε-greedy, as in Part 1). And every score starts at 0 while each step costs 1, so a move it has not tried yet looks better than one it has: untried moves get tried. Sutton and Barto (2018) call this *optimistic initial values*.

In [ ]:
def q_learning(trips, seed, alpha=0.5, epsilon=0.1):           # try changing alpha and epsilon
    rng = np.random.default_rng(seed)
    Q = {(cell, m): 0.0 for cell in cells for m in MOVES}
    route = []                                        # route length after each trip (None: no route to the goal)
    for _ in range(trips):
        cell = START
        for _ in range(100):
            options = list(MOVES)
            m = options[rng.integers(4)] if rng.random() < epsilon else greedy_move(cell, lambda c, mv: Q[(c, mv)])
            nxt, rew, done = step(cell, m)
            target = rew if done else rew + GAMMA * max(Q[(nxt, mv)] for mv in MOVES)
            Q[(cell, m)] += alpha * (target - Q[(cell, m)])
            cell = nxt
            if done: break
        steps, end = trip(lambda c: greedy_move(c, lambda cc, mv: Q[(cc, mv)]))
        route.append(steps if end == 'goal' else None)
    first = next((i + 1 for i, s in enumerate(route) if s == shortest), None)
    settled = next((i + 1 for i in range(len(route)) if all(s == shortest for s in route[i:])), None)
    return Q, first, settled

learned = {seed: q_learning(500, seed) for seed in range(10)}
first = [learned[s][1] for s in learned]; settled = [learned[s][2] for s in learned]
print('Ten separate learners, 500 trips each.')
print('Trip at which each first had a shortest route:   ', first)
print('Trip from which each kept one for good:          ', settled)
if ended != 'goal':
    print('\nThe best policy no longer reaches the goal, so there is no shortest route to compare. Look at the arrows instead.')
elif None in first:
    print('\nSome learners never found a shortest route with these settings.')
else:
    print(f'\nFirst found after {min(first)} to {max(first)} trips; kept for good from trip {min(settled)} to {max(settled)}.')
Q = learned[0][0]
print('\nWhat learner 0 ended with (best move on every square):')
show(lambda cell: greedy_move(cell, lambda c, mv: Q[(c, mv)]))
if SHIPPED_GRID:
    assert None not in settled, 'with the shipped settings every learner should keep a shortest route within 500 trips'
    print('Check passed: every learner ended with a shortest route.')
else:
    print('Check skipped: you changed the settings.')

**What to notice** (with the settings as shipped)

- **Same length, found a different way.** Value iteration *computed* a best route because it knew the rules. Q-learning *found* a shortest route from experience alone — not always the one value iteration printed, because several 8-step routes tie. Its first success was not always its last: most learners lost the route again for a few trips before keeping it.
- **Arrows away from the route may look odd.** The learner rarely visits squares far from its route, so its arrows there can be wrong. It learns well what it practices.
- **Knowing the rules is worth a great deal.** Learning by trial earns its place when the rules exist only as a simulator you can run but cannot write down as chances, or when there are far too many states to compute every value.
- **The reward is the whole instruction.** *Try:* set `STEP_COST = -5.0` and run Part 2 again. Where does the best policy send the robot now, and why? *Then try* `epsilon=0` in `q_learning`: does the robot still explore?

---
## Where to go next

- **Bigger problems.** This grid has 25 squares; a real problem can have more states than a table could hold. *Deep* RL replaces the table with a neural network that estimates the values (or the policy) from the state, so that similar states share what was learned.
- **Learning the policy directly.** Instead of scoring moves, some methods adjust the policy itself toward choices that paid off, and many do both: a policy, plus a value estimate to steady it. PPO (proximal policy optimization), used in the book's inventory chapter, is one of them.
- **In the book:** Appendix A.4 introduces the idea in a business setting. Sect. 3.4 applies it to reordering stock, and Sect. 5.4 to routing vehicles as orders and traffic change; its notebook uses the same Q-learning on a larger grid. Sect. 8.4 sets out how responses to disruptions could be learned inside a digital twin.
- **The field's textbook:** Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press. Its chapters on multi-armed bandits, dynamic programming, temporal-difference learning (where Q-learning is) and policy-gradient methods follow this notebook's order.

# ---
##How to cite

Luo, C. C. (2026). Reinforcement Learning: A First Look (Version 1.0.0) [Computer software]. Zenodo. https://doi.org/10.5281/zenodo.23968091

---
## License (code only)

The code cells of this notebook are under the MIT License, the same license as the book's code. The explanatory text is not.

```
MIT License

Copyright (c) 2026 Charles C. Luo

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```